In [ ]:
import akshare_proxy_patch
import os

_token = None
_env_path = os.path.join(os.path.dirname(os.path.abspath('.')), '.env')
for p in ['.env', _env_path]:
    if os.path.exists(p):
        with open(p) as f:
            for line in f:
                if line.strip().startswith('AKSHARE_PROXY_TOKEN='):
                    _token = line.strip().split('=', 1)[1]
                    break
        if _token:
            break
if not _token:
    raise RuntimeError("Missing AKSHARE_PROXY_TOKEN in .env file. Create .env with: AKSHARE_PROXY_TOKEN=your_token")

akshare_proxy_patch.install_patch(
    "101.201.173.125",
    auth_token=_token,
    retry=30,
    hook_domains=[
        "fund.eastmoney.com",
        "push2.eastmoney.com",
        "push2his.eastmoney.com",
        "emweb.securities.eastmoney.com",
    ],
)

import akshare as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import time, os
from datetime import datetime

matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False

CACHE_DIR = "cache_yf"
os.makedirs(CACHE_DIR, exist_ok=True)

etfs = {
    '沪深300':  '510300',
    'A500红利低波': '159339',
    '标普红利':  '562060',
    '恒生科技': '513130',
    '黄金ETF':  '518880',
    '十年国债': '511260',
    '城投ETF':  '511220',
}

end_date = datetime.today().strftime('%Y%m%d')
start_date = '20200101'

def load_etf(symbol, start, end, retries=3):
    """Load ETF daily data — use local cache if available and fresh (same end_date)."""
    cache_file = os.path.join(CACHE_DIR, f"{symbol}_{start}_{end}.csv")
    if os.path.exists(cache_file):
        close = pd.read_csv(cache_file, index_col=0, parse_dates=True).squeeze()
        close.name = symbol
        print(f"  {symbol}: loaded from cache ({len(close)} rows)")
        return close

    for attempt in range(retries):
        try:
            df = ak.fund_etf_hist_em(symbol=symbol, period="daily",
                                      start_date=start, end_date=end, adjust="qfq")
            if df is not None and not df.empty:
                df['日期'] = pd.to_datetime(df['日期'])
                df = df.set_index('日期').sort_index()
                close = df['收盘'].astype(float)
                close.name = symbol
                close.to_csv(cache_file)
                return close
        except Exception as e:
            print(f"  {symbol}: attempt {attempt+1} failed ({e})")
            if attempt < retries - 1:
                time.sleep(3 * (attempt + 1))
    return None

print(f"Loading data via AkShare ({start_date} to {end_date})...")
frames = {}
for name, symbol in etfs.items():
    series = load_etf(symbol, start_date, end_date)
    if series is not None and not series.empty:
        frames[name] = series
        print(f"  {name} ({symbol}): {len(series)} rows ({series.index.min().date()} to {series.index.max().date()})")
    else:
        print(f"  {name} ({symbol}): NO DATA")
    time.sleep(0.5)

data = pd.DataFrame(frames).dropna()
print(f"\nCombined data: {data.shape}, {data.index.min().date()} to {data.index.max().date()}")

print("\n=== 中证红利 (515080) Dividend History ===")
try:
    div = ak.fund_etf_dividend_sina(symbol="sh515080")
    if div is not None and not div.empty:
        print(div.to_string(index=False))
    else:
        print("  No dividend data available")
except Exception as e:
    print(f"  Could not fetch dividends: {e}")

In [ ]:
returns = data.pct_change().dropna()
trading_days = len(returns)
years = trading_days / 252

print(f"=== Individual ETF Performance ({data.index.min().date()} to {data.index.max().date()}, {years:.1f} years) ===\n")
stats = []
for col in data.columns:
    total_ret = (data[col].iloc[-1] / data[col].iloc[0] - 1) * 100
    ann_ret = ((data[col].iloc[-1] / data[col].iloc[0]) ** (1/years) - 1) * 100
    vol = returns[col].std() * np.sqrt(252) * 100
    sharpe = ann_ret / vol if vol > 0 else 0
    max_dd = ((data[col] / data[col].cummax()) - 1).min() * 100
    stats.append({'ETF': col, 'Total %': round(total_ret,1), 'Ann. %': round(ann_ret,1),
                  'Vol %': round(vol,1), 'Sharpe': round(sharpe,2), 'MaxDD %': round(max_dd,1)})

stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))

plt.figure(figsize=(14, 7))
for col in data.columns:
    normalized = data[col] / data[col].iloc[0] * 100
    plt.plot(normalized, label=col, linewidth=2)
plt.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
plt.title('Normalized ETF Prices (Base 100)')
plt.xlabel('Date')
plt.ylabel('Growth of ¥100')
plt.legend()
plt.grid()
plt.show()

In [ ]:
compare_etfs = {
    # Bonds
    '十年国债 (511260)': '511260',
    '城投ETF (511220)': '511220',
    '国债ETF (511010)': '511010',
    '短融ETF (511360)': '511360',
    '公司债ETF (511030)': '511030',
    '政金债ETF (511520)': '511520',
    # Dividend ETFs
    '中证红利 (515080)': '515080',
    '红利低波 (512890)': '512890',
    '红利低波100 (159307)': '159307',
    '红利质量 (159758)': '159758',
    'A500红利低波 (159339)': '159339',
    '标普红利 (562060)': '562060',
    '标普大盘红利低波50 (515450)': '515450',
}

print("Loading ETFs for comparison...")
cmp_data = {}
for name, sym in compare_etfs.items():
    series = load_etf(sym, start_date, end_date)
    if series is not None and not series.empty:
        cmp_data[name] = series
        print(f"  {name}: {len(series)} rows")
    else:
        print(f"  {name}: NO DATA")
    time.sleep(0.5)

cmp_df = pd.DataFrame(cmp_data).dropna()
cmp_ret = cmp_df.pct_change().dropna()
cmp_years = len(cmp_ret) / 252

print("\nFetching current market data...")
try:
    import io, requests
    spot_df = ak.fund_etf_spot_em()
    etf_mktval = {}
    for name, sym in compare_etfs.items():
        row = spot_df[spot_df['代码'] == sym]
        if not row.empty:
            mv = row.iloc[0].get('总市值', None)
            if mv is not None and mv > 0:
                etf_mktval[name] = mv / 1e8
            else:
                etf_mktval[name] = None
        else:
            etf_mktval[name] = None
except Exception as e:
    print(f"  Live market data unavailable ({e}), using known AUM estimates")
    etf_mktval = {
        '十年国债 (511260)': 108,
        '城投ETF (511220)': 62,
        '国债ETF (511010)': 81,
        '短融ETF (511360)': 230,
        '中证红利 (515080)': 320,
        '红利低波 (512890)': 125,
        '红利质量 (159758)': 42,
        '红利低波100 (159307)': 78,
    }

print(f"\n=== Bond & Dividend ETF Comparison ({cmp_df.index.min().date()} to {cmp_df.index.max().date()}, {cmp_years:.1f} years) ===\n")
print(f"{'ETF':>25s} {'市值(亿)':>8s} {'Ann%':>6s} {'Vol%':>6s} {'Sharpe':>7s} {'MaxDD%':>7s} {'Total%':>7s}")
print("-" * 75)
cmp_stats = []
for col in cmp_df.columns:
    total = (cmp_df[col].iloc[-1] / cmp_df[col].iloc[0] - 1) * 100
    ann = ((cmp_df[col].iloc[-1] / cmp_df[col].iloc[0]) ** (1/cmp_years) - 1) * 100
    vol = cmp_ret[col].std() * np.sqrt(252) * 100
    sr = ann / vol if vol > 0 else 0
    dd = ((cmp_df[col] / cmp_df[col].cummax()) - 1).min() * 100
    mv = etf_mktval.get(col)
    mv_str = f"{mv:>7.0f}" if mv is not None else "    N/A"
    print(f"{col:>25s} {mv_str} {ann:>6.1f} {vol:>6.1f} {sr:>7.2f} {dd:>7.1f} {total:>7.1f}")
    cmp_stats.append({'name': col, 'ann': ann, 'vol': vol, 'sharpe': sr, 'dd': dd, 'mktval': mv})

bond_names = [n for n in cmp_df.columns if any(k in n for k in ['国债', '城投', '短融', '公司债', '政金债'])]
div_names = [n for n in cmp_df.columns if any(k in n for k in ['红利', '中证'])]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for col in bond_names:
    norm = cmp_df[col] / cmp_df[col].iloc[0] * 100
    ax.plot(norm, label=col, linewidth=2)
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Bond ETFs: Normalized Prices (Base 100)')
ax.set_ylabel('Growth of ¥100')
ax.legend(fontsize=8)
ax.grid()

ax = axes[1]
for col in div_names:
    norm = cmp_df[col] / cmp_df[col].iloc[0] * 100
    ax.plot(norm, label=col, linewidth=2)
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Dividend ETFs: Normalized Prices (Base 100)')
ax.set_ylabel('Growth of ¥100')
ax.legend(fontsize=8)
ax.grid()

plt.tight_layout()
plt.show()

print("\n=== Correlation Matrix ===")
cmp_corr = cmp_ret.corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cmp_corr, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(cmp_corr)))
ax.set_xticklabels([n.split(' ')[0] for n in cmp_corr.columns], rotation=45, ha='right')
ax.set_yticks(range(len(cmp_corr)))
ax.set_yticklabels([n.split(' ')[0] for n in cmp_corr.columns])
for i in range(len(cmp_corr)):
    for j in range(len(cmp_corr)):
        ax.text(j, i, f'{cmp_corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im)
plt.title('Bond & Dividend ETF Correlation')
plt.tight_layout()
plt.show()

In [ ]:
corr = returns.corr()
print("=== Correlation Matrix ===\n")
print(corr.round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center')
fig.colorbar(im)
plt.title('Return Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
n_assets = len(data.columns)
mean_returns = returns.mean() * 252
cov_matrix = returns.cov() * 252

n_portfolios = 50000
min_weights = np.array([0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00])
max_weights = np.array([1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00])
results = np.zeros((4, n_portfolios))
all_weights = np.zeros((n_portfolios, n_assets))

np.random.seed(42)
count = 0
while count < n_portfolios:
    remaining = 1.0 - min_weights.sum()
    w = min_weights + np.random.dirichlet(np.ones(n_assets)) * remaining
    if np.all(w <= max_weights):
        ret = np.dot(w, mean_returns) * 100
        vol = np.sqrt(np.dot(w.T, np.dot(cov_matrix, w))) * 100
        results[0, count] = ret
        results[1, count] = vol
        results[2, count] = ret / vol if vol > 0 else 0
        all_weights[count] = w
        count += 1

max_sharpe_idx = results[2].argmax()
min_vol_idx = results[1].argmin()

plt.figure(figsize=(14, 8))
scatter = plt.scatter(results[1], results[0], c=results[2], cmap='viridis', alpha=0.3, s=5)
plt.colorbar(scatter, label='Sharpe Ratio')
plt.scatter(results[1, max_sharpe_idx], results[0, max_sharpe_idx], marker='*', color='red', s=300, label='Max Sharpe')
plt.scatter(results[1, min_vol_idx], results[0, min_vol_idx], marker='*', color='blue', s=300, label='Min Volatility')
plt.title('Efficient Frontier (50,000 Random Portfolios)')
plt.xlabel('Annualized Volatility %')
plt.ylabel('Annualized Return %')
plt.legend()
plt.grid()
plt.show()

print("=== Max Sharpe Ratio Portfolio ===")
print(f"  Expected Return: {results[0, max_sharpe_idx]:.1f}%")
print(f"  Volatility:      {results[1, max_sharpe_idx]:.1f}%")
print(f"  Sharpe Ratio:    {results[2, max_sharpe_idx]:.2f}")
print(f"  Allocation (¥100,000):")
for name, w in zip(data.columns, all_weights[max_sharpe_idx]):
    print(f"    {name}: {w*100:.1f}% (¥{w*100000:,.0f})")

print(f"\n=== Min Volatility Portfolio ===")
print(f"  Expected Return: {results[0, min_vol_idx]:.1f}%")
print(f"  Volatility:      {results[1, min_vol_idx]:.1f}%")
print(f"  Sharpe Ratio:    {results[2, min_vol_idx]:.2f}")
print(f"  Allocation (¥100,000):")
for name, w in zip(data.columns, all_weights[min_vol_idx]):
    print(f"    {name}: {w*100:.1f}% (¥{w*100000:,.0f})")

In [ ]:
target_returns = [5, 10, 15, 20]
tolerance = 0.3

print("=== Suggested Allocation by Target Return ===\n")
header = f"{'Target':>8s} {'Ret%':>6s} {'Vol%':>6s} {'Sharpe':>7s} {'MaxDD%':>7s}"
for col in data.columns:
    header += f" {col:>8s}"
print(header)
print("-" * len(header))

target_weights = {}
target_ret_actual = {}
target_dd_actual = {}
for target in target_returns:
    mask = np.abs(results[0] - target) < tolerance
    if mask.sum() == 0:
        print(f"{target:>7.0f}%  ** No portfolio found near this return **")
        continue
    candidates = np.where(mask)[0]
    best = candidates[results[1, candidates].argmin()]
    w = all_weights[best]
    target_weights[target] = w

    port_ret = (returns * w).sum(axis=1)
    cum = (1 + port_ret).cumprod() * 100
    max_dd = ((cum / cum.cummax()) - 1).min() * 100
    actual_ret = ((cum.iloc[-1] / 100) ** (1/years) - 1) * 100
    target_ret_actual[target] = actual_ret
    target_dd_actual[target] = max_dd

    row = f"{target:>7.0f}% {actual_ret:>6.1f} {results[1, best]:>6.1f} {results[2, best]:>7.2f} {max_dd:>7.1f}"
    for wi in w:
        row += f" {wi*100:>7.1f}%"
    print(row)

w_equal4 = np.zeros(n_assets)
for i, col in enumerate(data.columns):
    if col in ['A500红利低波', '黄金ETF', '十年国债', '城投ETF']:
        w_equal4[i] = 0.25
target_weights['EQ4'] = w_equal4
port_ret_eq4 = (returns * w_equal4).sum(axis=1)
cum_eq4 = (1 + port_ret_eq4).cumprod() * 100
ann_ret_eq4 = ((cum_eq4.iloc[-1] / 100) ** (1/years) - 1) * 100
vol_eq4 = port_ret_eq4.std() * np.sqrt(252) * 100
sharpe_eq4 = ann_ret_eq4 / vol_eq4 if vol_eq4 > 0 else 0
dd_eq4 = ((cum_eq4 / cum_eq4.cummax()) - 1).min() * 100
row = f"{'25%x4':>7s} {ann_ret_eq4:>6.1f} {vol_eq4:>6.1f} {sharpe_eq4:>7.2f} {dd_eq4:>7.1f}"
for wi in w_equal4:
    row += f" {wi*100:>7.1f}%"
print(row)

w_equal5 = np.zeros(n_assets)
for i, col in enumerate(data.columns):
    if col in ['A500红利低波', '标普红利', '黄金ETF', '十年国债', '城投ETF']:
        w_equal5[i] = 0.20
target_weights['EQ5'] = w_equal5
port_ret_eq5 = (returns * w_equal5).sum(axis=1)
cum_eq5 = (1 + port_ret_eq5).cumprod() * 100
ann_ret_eq5 = ((cum_eq5.iloc[-1] / 100) ** (1/years) - 1) * 100
vol_eq5 = port_ret_eq5.std() * np.sqrt(252) * 100
sharpe_eq5 = ann_ret_eq5 / vol_eq5 if vol_eq5 > 0 else 0
dd_eq5 = ((cum_eq5 / cum_eq5.cummax()) - 1).min() * 100
row = f"{'20%x5':>7s} {ann_ret_eq5:>6.1f} {vol_eq5:>6.1f} {sharpe_eq5:>7.2f} {dd_eq5:>7.1f}"
for wi in w_equal5:
    row += f" {wi*100:>7.1f}%"
print(row)

chart_weights = {k: v for k, v in target_weights.items()}

def fmt_label(t):
    if isinstance(t, (int, float)):
        ret = target_ret_actual.get(t, t)
        dd = target_dd_actual.get(t, 0)
        return f'{ret:.0f}% ret, {dd:.0f}% DD'
    elif t == 'EQ4':
        return f'25%x4 ({ann_ret_eq4:.0f}% ret, {dd_eq4:.0f}% DD)'
    elif t == 'EQ5':
        return f'20%x5 ({ann_ret_eq5:.0f}% ret, {dd_eq5:.0f}% DD)'
    return t

fig, ax = plt.subplots(figsize=(16, 6))
x = np.arange(len(chart_weights))
bottom = np.zeros(len(chart_weights))
colors = plt.cm.Set2(np.linspace(0, 1, n_assets))
etf_labels = {name: f"{name} ({sym})" for name, sym in etfs.items()}
for i, col in enumerate(data.columns):
    vals = [chart_weights[t][i] * 100 for t in chart_weights]
    ax.bar(x, vals, bottom=bottom, label=etf_labels[col], color=colors[i], width=0.6)
    for j, v in enumerate(vals):
        if v > 3:
            ax.text(x[j], bottom[j] + v/2, f'{v:.0f}%', ha='center', va='center', fontsize=8)
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels([fmt_label(t) for t in chart_weights], fontsize=8)
ax.set_ylabel('Weight %')
ax.set_title('Suggested Allocation by Target Annualized Return')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=8)
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()